In [1]:
import pandas as pd
import numpy as np
#Import StringIO for reading strings as files
from io import StringIO

annot_df = pd.read_csv('../gen_datasets/datasets/csa_annot.csv', sep='\t')
annot_df['sequence_len'] = annot_df['Sequence'].apply(lambda x: len(x) if isinstance(x, str) else 0)

In [2]:
import os
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

def hits_to_fasta(hit_df, out_dir, whitelist=None):
    def to_fasta(f, pid_l, seq_l):
        seq_records = [
            SeqRecord(Seq(seq), id=str(pid), description="")
            for pid, seq in zip(pid_l, seq_l)
        ]
        SeqIO.write(seq_records, f, "fasta")

    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    for group in hit_df.groupby('query'):
        query_id = group[0]
        if whitelist is None or query_id in whitelist:
            pass
            # print(f'Processing {query_id}')
            # print(query_id, group[1])
        else:
            continue
        # Filter for sequences with length within 2 standard deviations of the mean
        group_df = group[1]
        init_len = len(group_df)
        group_df = group_df[group_df['pident'] <= 95]
        m, std = np.mean(group_df['tlen']), np.std(group_df['tlen']) + 1
        group_df = group_df[(group_df['tlen'] > m - 2 * std) & (group_df['tlen'] < m + 2 * std)]
        if(len(group_df) < 5):
            print(group_df, query_id, init_len, m, std)
            continue
        group_id = list(group_df['subject'])
        group_seq = list(group_df['tseq'])
        
        if(len(group_id) > 450):
            group_id = group_id[:450]
            group_seq = group_seq[:450]
        if(not query_id in group_id):
            group_id.append(query_id)
            group_seq.append(group_df['qseq'].values[0])

        to_fasta(f'{out_dir}/{query_id}_homologues.fasta', group_id, group_seq)

In [3]:
hit_df = pd.read_csv('csa_aln.m8', sep='\t', header=None, 
                     names=['query', 'subject', 'fident', 'pident', 'nident', 'qlen', 'tlen', 
                            'qstart', 'qend', 'tstart', 'tend', 'evalue', 'bits', 'qseq', 'tseq'])
group_sizes = hit_df.groupby('query').size()
# whitelist = ['O76290', 'P13448:P13449', 'P56740', 'Q93EK7', 'Q967M2']
hits_to_fasta(hit_df, 'uniref_msa/csa_msa')

Empty DataFrame
Columns: [query, subject, fident, pident, nident, qlen, tlen, qstart, qend, tstart, tend, evalue, bits, qseq, tseq]
Index: [] P00962 303 nan nan
Empty DataFrame
Columns: [query, subject, fident, pident, nident, qlen, tlen, qstart, qend, tstart, tend, evalue, bits, qseq, tseq]
Index: [] P0AES2 348 nan nan
         query                 subject  fident  pident  nident  qlen  tlen  \
219494  P0AFG6  UniRef90_UPI000E2CC136   0.945    94.5     385   405   405   
219495  P0AFG6  UniRef90_UPI001A25F868   0.938    93.8     380   405   405   
219496  P0AFG6  UniRef90_UPI00209E2F8B   0.925    92.5     375   405   405   

        qstart  qend  tstart  tend         evalue  bits  \
219494       1   405       1   405  5.306000e-235   732   
219495       1   405       1   405  4.812000e-234   729   
219496       1   405       1   405  1.696000e-233   728   

                                                     qseq  \
219494  MSSVDILVPDLPESVADATVATWHKKPGDAVVRDEVLVEIETDKVV...   
219495

In [ ]:
hit_df = pd.read_csv('csa_aln.m8', sep='\t', header=None, 
                     names=['query', 'subject', 'fident', 'pident', 'nident', 'qlen', 'tlen', 
                            'qstart', 'qend', 'tstart', 'tend', 'evalue', 'bits', 'qseq', 'tseq'])
group_sizes = hit_df.groupby('query').size()
# whitelist = ['O76290', 'P13448:P13449', 'P56740', 'Q93EK7', 'Q967M2']
hits_to_fasta(hit_df, 'uniref_msa/csa_msa')

hit_df = pd.read_csv('llps_aln.m8', sep='\t', header=None, 
                     names=['query', 'subject', 'fident', 'pident', 'nident', 'qlen', 'tlen', 
                            'qstart', 'qend', 'tstart', 'tend', 'evalue', 'bits', 'qseq', 'tseq'])
hits_to_fasta(hit_df, 'uniref_msa/llps_msa')
hit_df = pd.read_csv('elms_aln.m8', sep='\t', header=None, 
                     names=['query', 'subject', 'fident', 'pident', 'nident', 'qlen', 'tlen', 
                            'qstart', 'qend', 'tstart', 'tend', 'evalue', 'bits', 'qseq', 'tseq'])
hits_to_fasta(hit_df, 'uniref_msa/elms_msa')

In [ ]:

hit_df['win_size'] = hit_df.apply(lambda x: x['tlen'] / x['qlen'] < 1.75, axis=1)
hit_df = hit_df[hit_df['win_size']]

def to_fasta(f, pid_l, seq_l):
    seq_records = [
        SeqRecord(Seq(seq), id=str(pid), description="")
        for pid, seq in zip(pid_l, seq_l)
    ]
    SeqIO.write(seq_records, f, "fasta")

import os
if not os.path.exists('uniref_msa/csa_msa'):
    os.makedirs('uniref_msa/csa_msa')
for group in hit_df.groupby('query'):
    query_id = group[0]
    group_df = group[1]
    
    group_id = list(group_df['subject'])
    group_seq = list(group_df['taln'])
    if(not query_id in group_id):
        group_id.append(query_id)
        group_seq.append(group_df['taln'].values[0])
    to_fasta(f'uniref_msa/csa_msa/{query_id}_homologues.fasta', group_id, group_seq)

In [ ]:
#query,target,fident,pident,nident,qlen,tlen,qstart,qend,tstart,tend,evalue,bits,qaln,taln
import Bio
from Bio import SeqIO
prot_rec = list(SeqIO.parse('swissprot_msa/uniprot_sprot.fasta', 'fasta'))
prot_id, prot_seq = [p.id for p in prot_rec], [str(p.seq) for p in prot_rec]
prot_df = pd.DataFrame({'id': prot_id, 'sequence': prot_seq})
hit_df = hit_df.merge(prot_df, left_on='subject', right_on='id', how='left')
annot_df['sequence_len'] = annot_df['Sequence'].apply(lambda x: len(x) if isinstance(x, str) else 0)
hit_df = hit_df.merge(annot_df[['UniprotID', 'sequence_len']], left_on='query', right_on='UniprotID', how='left')
hit_df.rename(columns={'sequence_len': 'query_len'}, inplace=True)
hit_df['sequence_len'] = hit_df['sequence'].apply(lambda x: len(x) if isinstance(x, str) else 0)

KeyError: 'tseq'

In [7]:
id_set = set(annot_df['UniprotID'])
print(len(id_set), 'unique Uniprot IDs in annotation file')
print(len(id_set.intersection(set(hit_df['subject']))), 'Uniprot IDs in annotation file that are also in the MSA hits')

846 unique Uniprot IDs in annotation file
750 Uniprot IDs in annotation file that are also in the MSA hits


In [2]:
hit_df = pd.read_csv('csa_matches.tsv', sep='\t', header=None, 
                     names=['query', 'subject', 'identity', 'length', 'mismatch', 'gapopen', 
                            'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore'])
import Bio
from Bio import SeqIO
prot_rec = list(SeqIO.parse('swissprot_msa/uniprot_sprot.fasta', 'fasta'))
prot_id, prot_seq = [p.id for p in prot_rec], [str(p.seq) for p in prot_rec]
prot_df = pd.DataFrame({'id': prot_id, 'sequence': prot_seq})
hit_df = hit_df.merge(prot_df, left_on='subject', right_on='id', how='left')
annot_df['sequence_len'] = annot_df['Sequence'].apply(lambda x: len(x) if isinstance(x, str) else 0)
hit_df = hit_df.merge(annot_df[['UniprotID', 'sequence_len']], left_on='query', right_on='UniprotID', how='left')
hit_df.rename(columns={'sequence_len': 'query_len'}, inplace=True)
hit_df['sequence_len'] = hit_df['sequence'].apply(lambda x: len(x) if isinstance(x, str) else 0)

In [ ]:
hit_df = hit_df[hit_df['sequence_len'] > 0]

In [ ]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
def to_fasta(f, pid_l, seq_l):
    seq_records = [
        SeqRecord(Seq(seq), id=str(pid), description="")
        for pid, seq in zip(pid_l, seq_l)
    ]
    SeqIO.write(seq_records, f, "fasta")

import os
if not os.path.exists('swissprot_msa/csa_msa'):
    os.makedirs('swissprot_msa/csa_msa')
for group in hit_df.groupby('query'):
    query_id = group[0]
    group_df = group[1]
    to_fasta(f'swissprot_msa/csa_msa/{query_id}_homologues.fasta', group_df['subject'], group_df['sequence'])

In [ ]:
import subprocess
def run_cd_hit(input_file, output_file, identity=0.95):
    try:
        subprocess.run(['../../cdhit/cd-hit', '-i', input_file, '-o', output_file, '-c', str(identity)], check=True)
        print(f"CD-HIT completed. Filtered sequences saved in {output_file}")
    except subprocess.CalledProcessError as e:
        print(f"Error during CD-HIT: {e}")

from io import StringIO
import requests

def add_seed(input_file, seed_id):
    existing_sequences = list(SeqIO.parse(input_file, "fasta"))
    if not any(seed_id in record.id for record in existing_sequences):
        try:
            fetched_sequence = fetch_protein_sequence(seed_id)
            fetched_seq_records = list(SeqIO.parse(StringIO(fetched_sequence), "fasta"))
            if not fetched_seq_records:
                raise ValueError(f"No sequence found for seed ID {seed_id}")

            fetched_seq_record = fetched_seq_records[0]
            with open(input_file, "a") as outfile:
                SeqIO.write([fetched_seq_record], outfile, "fasta")
            print(f"Seed {seed_id} added to {input_file}")
        except ValueError as e:
            print(f"Error fetching or adding seed sequence for {seed_id}: {e}")
            return False # failure to add seed
    else:
        print(f"Seed {seed_id} is already present in {input_file}")
    return True 

def filter_length(input_fasta, output_fasta, seed_id):
    fasta = list(SeqIO.parse(input_fasta, "fasta"))
    if not fasta:
        print(f"{input_fasta} is empty")
        return

    seed_len = None
    for record in fasta:
        if seed_id in record.id:
            seed_len = len(record.seq)
            break

    if seed_len is None:
        print(f"Reference sequence not found in {input_fasta}")
        return

    lengths = [len(seq.seq) for seq in fasta]
    two_SDs = np.std(lengths) * 2

    filtered_sequences = [seq for seq in fasta if abs(len(seq.seq) - seed_len) <= two_SDs]
    SeqIO.write(filtered_sequences, output_fasta, 'fasta')



25 A0QTN8
25 A2RJT9
25 A4XF23
25 A5JTM5
25 A5JUY8
2 E3PRJ5:E3PRJ4
1 O04147
25 O06644
7 O13297
7 O15527
25 O16025
8 O28603:O28604
25 O31156
25 O31168
19 O31243
25 O31465
25 O32727:O32728
3 O33199
25 O34508
5 O34714
25 O35543
25 O46427
18 O48917
25 O50152
25 O52942
25 O54050:O54051
6 O58403
4 O58727
24 O59413
25 O59791
5 O59893
25 O64411
8 O66188:O66187:O66186
1 O68557
5 O68884
25 O69762
25 O75164
25 O76290
25 O81192
25 O82827
9 O87172
25 O87948
21 O94766
25 P00175
8 P00183
25 P00327
25 P00334
25 P00341
25 P00349
25 P00374
23 P00381
3 P00383
25 P00388
25 P00390
25 P00392
25 P00431
25 P00433
25 P00435
3 P00436:P00437
25 P00445
23 P00452:P69924
25 P00455
25 P00469
25 P00480
23 P00484
25 P00488
25 P00489
21 P00491
25 P00509
25 P00517:P61926
25 P00518
25 P00573
16 P00586
3 P00588
25 P00590
25 P00592
17 P00634
25 P00636
23 P00639
13 P00644
10 P00646:P02984
25 P00698
4 P00720
25 P00722
25 P00727
25 P00730
2 P00733
25 P00747
25 P00750
25 P00766
25 P00782
25 P00803
25 P00805
6 P00806
25 P00816
2

In [38]:
import json
with open('/home/andrew/GO_interp/data/old_files/catalytic_residues_homologues.json', 'r') as f:
    csa_homologues = json.load(f)


In [ ]:
for entry in csa_homologues:
    

In [43]:
csa_homologues[0]

{'mcsa_id': 1,
 'roles_summary': 'activator, electrostatic stabiliser, hydrogen bond acceptor, hydrogen bond donor, proton acceptor',
 'function_location_abv': '',
 'ptm': '',
 'roles': [{'group_function': 'activator',
   'function_type': 'spectator',
   'function': 'activator',
   'emo': 'EMO_00038'},
  {'group_function': '',
   'function_type': 'interaction',
   'function': 'hydrogen bond acceptor',
   'emo': 'EMO_00113'},
  {'group_function': 'electrostatic interaction',
   'function_type': 'spectator',
   'function': 'electrostatic stabiliser',
   'emo': 'EMO_00033'},
  {'group_function': '',
   'function_type': 'interaction',
   'function': 'hydrogen bond donor',
   'emo': 'EMO_00114'},
  {'group_function': 'electrostatic interaction',
   'function_type': 'spectator',
   'function': 'electrostatic stabiliser',
   'emo': 'EMO_00033'},
  {'group_function': '',
   'function_type': 'interaction',
   'function': 'hydrogen bond donor',
   'emo': 'EMO_00114'},
  {'group_function': '',
  